In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models

# Configurando o hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando na: {device}")

# Resize gigante pro cifar10 funcionar na resnet
transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Preparando os dados...")
# diminui o bach size pra não explodir tudo
trainset_resnet = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform_resnet)
trainloader_resnet = torch.utils.data.DataLoader(trainset_resnet, batch_size=16, shuffle=True)

testset_resnet = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_resnet)
testloader_resnet = torch.utils.data.DataLoader(testset_resnet, batch_size=16, shuffle=False)

print("Dados prontos para uso!")

Rodando na: cpu
Preparando os dados...
Dados prontos para uso!


In [2]:
# detalhe importante: a resnet original foi treinada com imagens 224x224
# precisa dar esse resize gigante no cifar10 pra ela funcionar direito
transform_resnet = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# carrega os dados de treino e teste
trainset_resnet = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform_resnet)
# batch_size menor pq imagem 224x224 pesa mt na memoria
trainloader_resnet = torch.utils.data.DataLoader(trainset_resnet, batch_size=32, shuffle=True)

testset_resnet = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_resnet)
testloader_resnet = torch.utils.data.DataLoader(testset_resnet, batch_size=32, shuffle=False)

In [3]:
# baixa a arquitetura e os pesos da resnet18
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# ongela os pesos da rede pra nao treinar tudo do zero de novo
for param in resnet.parameters():
    param.requires_grad = False

# pega o numero de neuronios que entra na ultima camada
num_ftrs = resnet.fc.in_features

# troca a ultima camada pra ter so 10 saidas (as classes do cifar)
resnet.fc = nn.Linear(num_ftrs, 10)

resnet = resnet.to(device)

criterion = nn.CrossEntropyLoss()
# so passa os parametros da ultima camada pro otimizador!
optimizer_resnet = optim.Adam(resnet.fc.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\mateu/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:02<00:00, 21.6MB/s]


In [4]:
epocas_resnet = 3 # 3 epocas ja ta otimo pq ela ja eh pre-treinada
historico_loss_resnet = []

resnet.train()

print("Treinando a ultima camada da ResNet no CIFAR10...")
for epoch in range(epocas_resnet):
    loss_acumulada = 0.0
    for inputs, labels in trainloader_resnet:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_resnet.zero_grad()

        outputs = resnet(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_resnet.step()

        loss_acumulada += loss.item()
    
    loss_media = loss_acumulada / len(trainloader_resnet)
    historico_loss_resnet.append(loss_media)
    print(f"Época {epoch+1}/{epocas_resnet} - Loss: {loss_media:.4f}")

Treinando a ultima camada da ResNet no CIFAR10...
Época 1/3 - Loss: 0.8013
Época 2/3 - Loss: 0.6377
Época 3/3 - Loss: 0.6147


In [ ]:
resnet.eval()
acertos = 0
total = 0

print("Calculando acuracia da ResNet, aguarde (demora um pouco por causa do tamanho das imagens)...")
with torch.no_grad():
    for inputs, labels in testloader_resnet:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = resnet(inputs)
        _, previsao = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        acertos += (previsao == labels).sum().item()
        
acc_resnet = 100 * acertos / total
print(f"Acurácia da ResNet18 no CIFAR10: {acc_resnet:.2f}%")

Calculando acuracia da ResNet, aguarde (demora um pouco por causa do tamanho das imagens)...
Acurácia da ResNet18 no CIFAR10: 80.01%


: 